
# Análisis gráfico de log Linux `auth.csv`

Este notebook carga el dataset `auth.csv`, genera gráficos de análisis de seguridad, los muestra dentro del notebook y también los guarda automáticamente en un directorio llamado `graficos`.

El dataset contiene eventos típicos de `auth.log` de Linux, como accesos SSH, comandos `sudo`, sesiones, usuarios, IPs de origen, tipo de evento, tipo de ataque y etiqueta de ataque.


In [ ]:

# ============================================================
# 1. IMPORTACIÓN DE LIBRERÍAS
# ============================================================

import os
import re
import warnings
from pathlib import Path

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')

# Configuración general de pandas
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 120)

# Configuración general de matplotlib
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['font.size'] = 10


In [ ]:

# ============================================================
# 2. CARGA DEL DATASET
# ============================================================

archivo = 'auth.csv'
df = pd.read_csv(archivo)

print('Dimensiones del dataset:', df.shape)
print('
Columnas detectadas:')
print(df.columns.tolist())

df.head()


In [ ]:

# ============================================================
# 3. CREACIÓN DEL DIRECTORIO PARA GUARDAR LOS GRÁFICOS
# ============================================================

directorio_graficos = Path('graficos')
directorio_graficos.mkdir(exist_ok=True)

print(f'Directorio creado/verificado: {directorio_graficos.resolve()}')


In [ ]:

# ============================================================
# 4. LIMPIEZA Y PREPARACIÓN DE DATOS
# ============================================================

# Copia de trabajo para no modificar directamente el dataframe original.
datos = df.copy()

# Normalización de nombres de columnas.
datos.columns = [c.strip() for c in datos.columns]

# Conversión de la columna fecha.
# El formato de auth.log normalmente no trae año: ejemplo "May 04 19:16:46".
# Se agrega un año de referencia para poder trabajar con pandas datetime.
if 'fecha' in datos.columns:
    anio_referencia = 2026
    datos['fecha_original'] = datos['fecha']
    datos['fecha_dt'] = pd.to_datetime(
        datos['fecha'].astype(str) + f' {anio_referencia}',
        format='%b %d %H:%M:%S %Y',
        errors='coerce'
    )
    datos['dia'] = datos['fecha_dt'].dt.date
    datos['hora'] = datos['fecha_dt'].dt.hour
    datos['minuto'] = datos['fecha_dt'].dt.minute
    datos['dia_semana'] = datos['fecha_dt'].dt.day_name()
    datos['fecha_hora'] = datos['fecha_dt'].dt.floor('H')

# Conversión de etiqueta de ataque.
if 'es_ataque' in datos.columns:
    datos['es_ataque'] = pd.to_numeric(datos['es_ataque'], errors='coerce').fillna(0).astype(int)
    datos['clasificacion'] = np.where(datos['es_ataque'] == 1, 'Ataque', 'Normal')

# Relleno básico para categorías vacías.
for col in ['host', 'usuario', 'ip_origen', 'evento', 'tipo_ataque', 'mensaje_original']:
    if col in datos.columns:
        datos[col] = datos[col].fillna('No disponible').astype(str)

print('Resumen de datos preparados:')
display(datos.head())
print('
Valores nulos por columna:')
display(datos.isna().sum())


In [ ]:

# ============================================================
# 5. FUNCIÓN PARA MOSTRAR Y GUARDAR GRÁFICOS
# ============================================================

def guardar_mostrar(nombre_archivo):
    '''
    Guarda el gráfico actual en el directorio graficos y luego lo muestra.
    
    Parámetros:
    nombre_archivo: nombre del archivo PNG que se guardará.
    '''
    ruta = directorio_graficos / nombre_archivo
    plt.tight_layout()
    plt.savefig(ruta, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Gráfico guardado en: {ruta}')


def grafico_barras_serie(serie, titulo, xlabel, ylabel, nombre_archivo, top=None, rotacion=45):
    '''
    Crea un gráfico de barras a partir de una serie de pandas.
    '''
    if top:
        serie = serie.head(top)
    plt.figure(figsize=(12, 6))
    serie.plot(kind='bar')
    plt.title(titulo)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.xticks(rotation=rotacion, ha='right')
    guardar_mostrar(nombre_archivo)



## 6. Exploración general del dataset


In [ ]:

# Información general del dataset
print('Información general:')
print(datos.info())

print('
Descripción estadística:')
display(datos.describe(include='all'))



## 7. Gráficos del log de autenticación Linux

Cada gráfico se muestra en pantalla y se guarda automáticamente dentro del directorio `graficos`.


In [ ]:
# 1. Distribución de eventos normales vs ataques
conteo = datos['clasificacion'].value_counts()
grafico_barras_serie(conteo, 'Distribución de eventos normales vs ataques', 'Clasificación', 'Cantidad de eventos', '01_distribucion_normal_vs_ataque.png', rotacion=0)

In [ ]:
# 2. Porcentaje de eventos normales vs ataques
plt.figure(figsize=(8, 8))
datos['clasificacion'].value_counts().plot(kind='pie', autopct='%1.1f%%', startangle=90)
plt.title('Porcentaje de eventos normales vs ataques')
plt.ylabel('')
guardar_mostrar('02_porcentaje_normal_vs_ataque.png')

In [ ]:
# 3. Eventos por tipo de ataque
conteo = datos['tipo_ataque'].value_counts()
grafico_barras_serie(conteo, 'Eventos por tipo de ataque', 'Tipo de ataque', 'Cantidad', '03_eventos_por_tipo_ataque.png', top=20)

In [ ]:
# 4. Eventos por categoría de evento
conteo = datos['evento'].value_counts()
grafico_barras_serie(conteo, 'Eventos por categoría de evento', 'Evento', 'Cantidad', '04_eventos_por_categoria.png', top=20)

In [ ]:
# 5. Eventos por host
conteo = datos['host'].value_counts()
grafico_barras_serie(conteo, 'Eventos por host', 'Host', 'Cantidad', '05_eventos_por_host.png', top=20)

In [ ]:
# 6. Top usuarios con más eventos
conteo = datos[datos['usuario'] != 'No disponible']['usuario'].value_counts()
grafico_barras_serie(conteo, 'Top usuarios con más eventos', 'Usuario', 'Cantidad', '06_top_usuarios_eventos.png', top=20)

In [ ]:
# 7. Top IPs de origen con más eventos
conteo = datos[datos['ip_origen'] != 'No disponible']['ip_origen'].value_counts()
grafico_barras_serie(conteo, 'Top IPs de origen con más eventos', 'IP de origen', 'Cantidad', '07_top_ips_eventos.png', top=20)

In [ ]:
# 8. Top IPs con más ataques
ataques = datos[(datos['es_ataque'] == 1) & (datos['ip_origen'] != 'No disponible')]
conteo = ataques['ip_origen'].value_counts()
grafico_barras_serie(conteo, 'Top IPs de origen con más ataques', 'IP de origen', 'Cantidad de ataques', '08_top_ips_ataques.png', top=20)

In [ ]:
# 9. Top usuarios asociados a ataques
ataques = datos[(datos['es_ataque'] == 1) & (datos['usuario'] != 'No disponible')]
conteo = ataques['usuario'].value_counts()
grafico_barras_serie(conteo, 'Top usuarios asociados a ataques', 'Usuario', 'Cantidad de ataques', '09_top_usuarios_ataques.png', top=20)

In [ ]:
# 10. Eventos por día
conteo = datos.groupby('dia').size()
plt.figure(figsize=(12, 6))
conteo.plot(kind='line', marker='o')
plt.title('Eventos por día')
plt.xlabel('Día')
plt.ylabel('Cantidad de eventos')
guardar_mostrar('10_eventos_por_dia.png')

In [ ]:
# 11. Ataques por día
conteo = datos[datos['es_ataque'] == 1].groupby('dia').size()
plt.figure(figsize=(12, 6))
conteo.plot(kind='line', marker='o')
plt.title('Ataques por día')
plt.xlabel('Día')
plt.ylabel('Cantidad de ataques')
guardar_mostrar('11_ataques_por_dia.png')

In [ ]:
# 12. Eventos por hora del día
conteo = datos['hora'].value_counts().sort_index()
plt.figure(figsize=(12, 6))
conteo.plot(kind='bar')
plt.title('Eventos por hora del día')
plt.xlabel('Hora')
plt.ylabel('Cantidad de eventos')
plt.xticks(rotation=0)
guardar_mostrar('12_eventos_por_hora.png')

In [ ]:
# 13. Ataques por hora del día
conteo = datos[datos['es_ataque'] == 1]['hora'].value_counts().sort_index()
plt.figure(figsize=(12, 6))
conteo.plot(kind='bar')
plt.title('Ataques por hora del día')
plt.xlabel('Hora')
plt.ylabel('Cantidad de ataques')
plt.xticks(rotation=0)
guardar_mostrar('13_ataques_por_hora.png')

In [ ]:
# 14. Eventos por día de la semana
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
conteo = datos['dia_semana'].value_counts().reindex(orden_dias).dropna()
plt.figure(figsize=(12, 6))
conteo.plot(kind='bar')
plt.title('Eventos por día de la semana')
plt.xlabel('Día de la semana')
plt.ylabel('Cantidad de eventos')
plt.xticks(rotation=45, ha='right')
guardar_mostrar('14_eventos_por_dia_semana.png')

In [ ]:
# 15. Ataques por día de la semana
orden_dias = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
conteo = datos[datos['es_ataque'] == 1]['dia_semana'].value_counts().reindex(orden_dias).dropna()
plt.figure(figsize=(12, 6))
conteo.plot(kind='bar')
plt.title('Ataques por día de la semana')
plt.xlabel('Día de la semana')
plt.ylabel('Cantidad de ataques')
plt.xticks(rotation=45, ha='right')
guardar_mostrar('15_ataques_por_dia_semana.png')

In [ ]:
# 16. Serie temporal por hora
conteo = datos.groupby('fecha_hora').size()
plt.figure(figsize=(14, 6))
conteo.plot(kind='line')
plt.title('Serie temporal de eventos por hora')
plt.xlabel('Fecha y hora')
plt.ylabel('Cantidad de eventos')
guardar_mostrar('16_serie_temporal_eventos_por_hora.png')

In [ ]:
# 17. Serie temporal de ataques por hora
conteo = datos[datos['es_ataque'] == 1].groupby('fecha_hora').size()
plt.figure(figsize=(14, 6))
conteo.plot(kind='line')
plt.title('Serie temporal de ataques por hora')
plt.xlabel('Fecha y hora')
plt.ylabel('Cantidad de ataques')
guardar_mostrar('17_serie_temporal_ataques_por_hora.png')

In [ ]:
# 18. Eventos normales vs ataques por hora
pivot = pd.crosstab(datos['hora'], datos['clasificacion'])
plt.figure(figsize=(12, 6))
pivot.plot(kind='bar', stacked=True, figsize=(12, 6))
plt.title('Eventos normales vs ataques por hora')
plt.xlabel('Hora')
plt.ylabel('Cantidad de eventos')
plt.xticks(rotation=0)
guardar_mostrar('18_normal_vs_ataque_por_hora.png')

In [ ]:
# 19. Eventos normales vs ataques por tipo de evento
pivot = pd.crosstab(datos['evento'], datos['clasificacion']).sort_values(by=pivot.columns.tolist(), ascending=False) if False else pd.crosstab(datos['evento'], datos['clasificacion'])
pivot['Total'] = pivot.sum(axis=1)
pivot = pivot.sort_values('Total', ascending=False).drop(columns='Total').head(15)
plt.figure(figsize=(14, 7))
pivot.plot(kind='bar', stacked=True, figsize=(14, 7))
plt.title('Eventos normales vs ataques por tipo de evento')
plt.xlabel('Tipo de evento')
plt.ylabel('Cantidad')
plt.xticks(rotation=45, ha='right')
guardar_mostrar('19_normal_vs_ataque_por_evento.png')

In [ ]:
# 20. Tipos de ataque por hora
pivot = pd.crosstab(datos[datos['es_ataque'] == 1]['hora'], datos[datos['es_ataque'] == 1]['tipo_ataque'])
plt.figure(figsize=(14, 7))
pivot.plot(kind='bar', stacked=True, figsize=(14, 7))
plt.title('Tipos de ataque por hora')
plt.xlabel('Hora')
plt.ylabel('Cantidad de ataques')
plt.xticks(rotation=0)
guardar_mostrar('20_tipos_ataque_por_hora.png')

In [ ]:
# 21. Heatmap: ataques por día y hora
pivot = datos[datos['es_ataque'] == 1].pivot_table(index='dia', columns='hora', values='es_ataque', aggfunc='count', fill_value=0)
plt.figure(figsize=(14, 6))
plt.imshow(pivot, aspect='auto')
plt.title('Heatmap de ataques por día y hora')
plt.xlabel('Hora')
plt.ylabel('Día')
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.colorbar(label='Cantidad de ataques')
guardar_mostrar('21_heatmap_ataques_dia_hora.png')

In [ ]:
# 22. Heatmap: eventos por día y hora
pivot = datos.pivot_table(index='dia', columns='hora', values='evento', aggfunc='count', fill_value=0)
plt.figure(figsize=(14, 6))
plt.imshow(pivot, aspect='auto')
plt.title('Heatmap de eventos por día y hora')
plt.xlabel('Hora')
plt.ylabel('Día')
plt.xticks(range(len(pivot.columns)), pivot.columns)
plt.yticks(range(len(pivot.index)), pivot.index)
plt.colorbar(label='Cantidad de eventos')
guardar_mostrar('22_heatmap_eventos_dia_hora.png')

In [ ]:
# 23. Matriz tipo de ataque vs evento
pivot = pd.crosstab(datos['tipo_ataque'], datos['evento'])
pivot = pivot.loc[pivot.sum(axis=1).sort_values(ascending=False).head(10).index]
plt.figure(figsize=(14, 7))
plt.imshow(pivot, aspect='auto')
plt.title('Matriz tipo de ataque vs evento')
plt.xlabel('Evento')
plt.ylabel('Tipo de ataque')
plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=45, ha='right')
plt.yticks(range(len(pivot.index)), pivot.index)
plt.colorbar(label='Cantidad')
guardar_mostrar('23_matriz_tipo_ataque_vs_evento.png')

In [ ]:
# 24. Matriz usuario vs tipo de ataque
ataques = datos[(datos['es_ataque'] == 1) & (datos['usuario'] != 'No disponible')]
top_usuarios = ataques['usuario'].value_counts().head(15).index
pivot = pd.crosstab(ataques[ataques['usuario'].isin(top_usuarios)]['usuario'], ataques[ataques['usuario'].isin(top_usuarios)]['tipo_ataque'])
plt.figure(figsize=(14, 7))
plt.imshow(pivot, aspect='auto')
plt.title('Matriz usuario vs tipo de ataque')
plt.xlabel('Tipo de ataque')
plt.ylabel('Usuario')
plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=45, ha='right')
plt.yticks(range(len(pivot.index)), pivot.index)
plt.colorbar(label='Cantidad')
guardar_mostrar('24_matriz_usuario_vs_tipo_ataque.png')

In [ ]:
# 25. Matriz IP vs tipo de ataque
data_ip = datos[(datos['es_ataque'] == 1) & (datos['ip_origen'] != 'No disponible')]
top_ips = data_ip['ip_origen'].value_counts().head(15).index
pivot = pd.crosstab(data_ip[data_ip['ip_origen'].isin(top_ips)]['ip_origen'], data_ip[data_ip['ip_origen'].isin(top_ips)]['tipo_ataque'])
plt.figure(figsize=(14, 7))
plt.imshow(pivot, aspect='auto')
plt.title('Matriz IP de origen vs tipo de ataque')
plt.xlabel('Tipo de ataque')
plt.ylabel('IP de origen')
plt.xticks(range(len(pivot.columns)), pivot.columns, rotation=45, ha='right')
plt.yticks(range(len(pivot.index)), pivot.index)
plt.colorbar(label='Cantidad')
guardar_mostrar('25_matriz_ip_vs_tipo_ataque.png')

In [ ]:
# 26. Top mensajes originales más repetidos
conteo = datos['mensaje_original'].value_counts().head(15)
plt.figure(figsize=(14, 7))
conteo.sort_values().plot(kind='barh')
plt.title('Top mensajes originales más repetidos')
plt.xlabel('Cantidad')
plt.ylabel('Mensaje original')
guardar_mostrar('26_top_mensajes_originales.png')

In [ ]:
# 27. Longitud de mensajes originales
if 'mensaje_original' in datos.columns:
    datos['longitud_mensaje'] = datos['mensaje_original'].str.len()
    plt.figure(figsize=(12, 6))
    datos['longitud_mensaje'].hist(bins=30)
    plt.title('Distribución de longitud de mensajes originales')
    plt.xlabel('Longitud del mensaje')
    plt.ylabel('Frecuencia')
    guardar_mostrar('27_distribucion_longitud_mensaje.png')

In [ ]:
# 28. Longitud del mensaje por clasificación
if 'longitud_mensaje' not in datos.columns:
    datos['longitud_mensaje'] = datos['mensaje_original'].str.len()
plt.figure(figsize=(10, 6))
datos.boxplot(column='longitud_mensaje', by='clasificacion')
plt.title('Longitud del mensaje por clasificación')
plt.suptitle('')
plt.xlabel('Clasificación')
plt.ylabel('Longitud del mensaje')
guardar_mostrar('28_boxplot_longitud_mensaje_por_clasificacion.png')

In [ ]:
# 29. Porcentaje de ataques por tipo de evento
resumen = datos.groupby('evento')['es_ataque'].mean().sort_values(ascending=False).head(20) * 100
plt.figure(figsize=(12, 6))
resumen.plot(kind='bar')
plt.title('Porcentaje de ataques por tipo de evento')
plt.xlabel('Tipo de evento')
plt.ylabel('% de eventos clasificados como ataque')
plt.xticks(rotation=45, ha='right')
guardar_mostrar('29_porcentaje_ataques_por_evento.png')

In [ ]:
# 30. Usuarios únicos por hora
usuarios_validos = datos[datos['usuario'] != 'No disponible']
conteo = usuarios_validos.groupby('hora')['usuario'].nunique()
plt.figure(figsize=(12, 6))
conteo.plot(kind='bar')
plt.title('Usuarios únicos por hora')
plt.xlabel('Hora')
plt.ylabel('Cantidad de usuarios únicos')
plt.xticks(rotation=0)
guardar_mostrar('30_usuarios_unicos_por_hora.png')

In [ ]:
# 31. IPs únicas por hora
ips_validas = datos[datos['ip_origen'] != 'No disponible']
conteo = ips_validas.groupby('hora')['ip_origen'].nunique()
plt.figure(figsize=(12, 6))
conteo.plot(kind='bar')
plt.title('IPs únicas por hora')
plt.xlabel('Hora')
plt.ylabel('Cantidad de IPs únicas')
plt.xticks(rotation=0)
guardar_mostrar('31_ips_unicas_por_hora.png')

In [ ]:
# 32. Ranking de combinaciones usuario + IP en ataques
ataques = datos[(datos['es_ataque'] == 1) & (datos['usuario'] != 'No disponible') & (datos['ip_origen'] != 'No disponible')]
if not ataques.empty:
    combinaciones = ataques.groupby(['usuario', 'ip_origen']).size().sort_values(ascending=False).head(20)
    etiquetas = [f'{u} / {ip}' for u, ip in combinaciones.index]
    plt.figure(figsize=(14, 7))
    plt.barh(etiquetas[::-1], combinaciones.values[::-1])
    plt.title('Top combinaciones usuario + IP en ataques')
    plt.xlabel('Cantidad de ataques')
    plt.ylabel('Usuario / IP')
    guardar_mostrar('32_top_usuario_ip_ataques.png')
else:
    print('No hay combinaciones usuario + IP para ataques.')

In [ ]:
# 33. Actividad acumulada de ataques
ataques_tiempo = datos[datos['es_ataque'] == 1].sort_values('fecha_dt')
if not ataques_tiempo.empty:
    ataques_tiempo['ataques_acumulados'] = range(1, len(ataques_tiempo) + 1)
    plt.figure(figsize=(14, 6))
    plt.plot(ataques_tiempo['fecha_dt'], ataques_tiempo['ataques_acumulados'])
    plt.title('Actividad acumulada de ataques')
    plt.xlabel('Tiempo')
    plt.ylabel('Ataques acumulados')
    guardar_mostrar('33_actividad_acumulada_ataques.png')
else:
    print('No hay ataques registrados.')

In [ ]:
# 34. Ataques por host y tipo de ataque
ataques = datos[datos['es_ataque'] == 1]
pivot = pd.crosstab(ataques['host'], ataques['tipo_ataque'])
plt.figure(figsize=(14, 7))
pivot.plot(kind='bar', stacked=True, figsize=(14, 7))
plt.title('Ataques por host y tipo de ataque')
plt.xlabel('Host')
plt.ylabel('Cantidad de ataques')
plt.xticks(rotation=45, ha='right')
guardar_mostrar('34_ataques_por_host_tipo_ataque.png')

In [ ]:
# 35. Eventos sospechosos por minuto
ataques = datos[datos['es_ataque'] == 1]
conteo = ataques['minuto'].value_counts().sort_index()
plt.figure(figsize=(12, 6))
conteo.plot(kind='bar')
plt.title('Ataques por minuto')
plt.xlabel('Minuto')
plt.ylabel('Cantidad de ataques')
plt.xticks(rotation=0)
guardar_mostrar('35_ataques_por_minuto.png')


## 8. Resumen ejecutivo automático


In [ ]:

# ============================================================
# RESUMEN AUTOMÁTICO DEL DATASET
# ============================================================

total_eventos = len(datos)
total_ataques = int(datos['es_ataque'].sum()) if 'es_ataque' in datos.columns else 0
total_normales = total_eventos - total_ataques
porcentaje_ataques = (total_ataques / total_eventos * 100) if total_eventos else 0

print('================ RESUMEN EJECUTIVO DEL LOG AUTH ================')
print(f'Total de eventos analizados: {total_eventos}')
print(f'Total de eventos normales: {total_normales}')
print(f'Total de eventos clasificados como ataque: {total_ataques}')
print(f'Porcentaje de ataques: {porcentaje_ataques:.2f}%')

if 'tipo_ataque' in datos.columns:
    print('
Top tipos de ataque:')
    display(datos[datos['es_ataque'] == 1]['tipo_ataque'].value_counts().head(10))

if 'ip_origen' in datos.columns:
    print('
Top IPs de origen asociadas a ataques:')
    display(datos[(datos['es_ataque'] == 1) & (datos['ip_origen'] != 'No disponible')]['ip_origen'].value_counts().head(10))

if 'usuario' in datos.columns:
    print('
Top usuarios asociados a ataques:')
    display(datos[(datos['es_ataque'] == 1) & (datos['usuario'] != 'No disponible')]['usuario'].value_counts().head(10))

print('
Todos los gráficos fueron guardados en el directorio:', directorio_graficos.resolve())
